In [ ]:
# Parameters
RESULTS_DIR = 'results'   # directory with *.jsonl results
OUTPUT_DIR = 'plots'      # where to save figures/CSVs

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline


In [ ]:
def plot_causal_accuracy(results_by_probe: dict):
    if not results_by_probe:
        print('No causal results found')
        return
    plt.figure(figsize=(14, 8))
    colors = {
        'mean': 'blue',
        'max': 'red',
        'rolling_means': 'green',
        'softmax': 'orange',
        'attention': 'purple'
    }
    for probe, df in results_by_probe.items():
        if df.empty:
            continue
        plt.plot(df['bin_idx'], df['accuracy'], marker='o', label=probe, color=colors.get(probe, 'gray'), linewidth=2, markersize=4)
    plt.axhline(y=50.0, color='black', linestyle='--', linewidth=1, alpha=0.7, label='random baseline (50%)')
    plt.xlabel('Position Bin Index (1% bins)')
    plt.ylabel('Test Accuracy (%)')
    plt.title('Causal Probe Accuracy vs Position (Layer 60)')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

plot_causal_accuracy(results_by_probe)


In [ ]:
def save_results_csv(results_by_probe: dict, out_path: str = f"{OUTPUT_DIR}/causal_results.csv"):
    rows = []
    for probe, df in results_by_probe.items():
        for _, r in df.iterrows():
            rows.append({'probe_type': probe, 'bin_idx': int(r['bin_idx']), 'accuracy': float(r['accuracy']), 'run_name': r.get('run_name', '')})
    out = pd.DataFrame(rows).sort_values(['probe_type', 'bin_idx'])
    out.to_csv(out_path, index=False)
    return out

csv_df = save_results_csv(results_by_probe)
csv_df.head()


In [ ]:
# Early vs Late position comparison
from itertools import product

def plot_early_late(results_by_probe: dict):
    if not results_by_probe:
        return
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    # Early bins 0-20
    for probe, df in results_by_probe.items():
        early = df[df['bin_idx'] <= 20]
        if not early.empty:
            ax1.plot(early['bin_idx'], early['accuracy'], marker='o', label=probe)
    ax1.set_title('Early Positions (0-20%)')
    ax1.set_xlabel('Bin')
    ax1.set_ylabel('Accuracy (%)')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    # Late bins 80-100
    for probe, df in results_by_probe.items():
        late = df[df['bin_idx'] >= 80]
        if not late.empty:
            ax2.plot(late['bin_idx'], late['accuracy'], marker='o', label=probe)
    ax2.set_title('Late Positions (80-100%)')
    ax2.set_xlabel('Bin')
    ax2.set_ylabel('Accuracy (%)')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    plt.tight_layout()

plot_early_late(results_by_probe)
